# build_omnicontext
Builds the OmniContext benchmark as a local artifact and reports what it contains. Pins the dataset revision, extracts every reference image through `exif_transpose` in the order `omnicontext/inference.py` uses, and writes a manifest of relative paths. The layout mirrors the domain set: a manifest jsonl plus an image root, so the generation and scoring notebooks read it exactly as they read `test.jsonl` and `imgs`.

Reports the facts the arm depends on: row and subtask counts, the field names the official scorer reads, which rows take `with_scene=True`, instruction language, EXIF orientation on the source images, token budget against the 1024-row conditioning cap, reference-count mix, and how many cases fall inside the consumer-electronics domain.

Write-once on the manifest, in the shape of `data` and `build_fewshot_exemplars`: the five generation runs and five scoring runs key on these paths and this row order, so a rebuild would repoint every one of them.

CPU runtime, no compute units. Writes `omnicontext/manifest.jsonl`, `omnicontext/images/`, `omnicontext/revision.txt`.

## 1. Drive, revision pin, download or restore

In [1]:
from google.colab import drive
import os, sys, json, subprocess
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
OC    = f"{DRIVE}/omnicontext"
IMGS  = f"{OC}/images"
MANIFEST = f"{OC}/manifest.jsonl"
REVFILE  = f"{OC}/revision.txt"
REPO_ID  = "OmniGen2/OmniContext"
os.makedirs(IMGS, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
from huggingface_hub import HfApi
from datasets import load_dataset, get_dataset_config_names

# Pin the dataset the way the code is pinned to 18e6f9d. First run records the current
# revision; later runs assert against it, so a silent upstream edit cannot change the
# benchmark between the baseline session and a finalist session.
head = HfApi().dataset_info(REPO_ID).sha
if os.path.exists(REVFILE):
    REVISION = open(REVFILE).read().strip()
    if REVISION != head:
        print(f"WARNING upstream moved: pinned {REVISION}, hub head {head}. Pinned revision kept.")
else:
    REVISION = head
    open(REVFILE, "w").write(REVISION + "\n")
print("dataset revision pinned:", REVISION)

CONFIGS = sorted(get_dataset_config_names(REPO_ID, revision=REVISION))
print(f"configs ({len(CONFIGS)}):", CONFIGS)

Mounted at /content/drive
dataset revision pinned: 0f6aabbe8d3448d51184093372e980205d9980d3


README.md:   0%|          | 0.00/4.16k [00:00<?, ?B/s]

configs (1): ['default']


## 2. Extract to files, write the manifest
Write-once. The five generation sessions and five scoring sessions key on these paths; regenerating them would repoint every run.

In [2]:
import collections
from PIL import ImageOps

orient = collections.Counter()   # EXIF tag 274 on the source images, counted before transposing

if os.path.exists(MANIFEST):
    rows = [json.loads(l) for l in open(MANIFEST, encoding="utf-8")]
    print(f"manifest exists, kept: {len(rows)} rows")
else:
    rows = []
    for cfg in CONFIGS:
        ds = load_dataset(REPO_ID, cfg, split="train", revision=REVISION)
        for item in ds:
            key = item["key"]
            paths, sizes = [], []
            for j, im in enumerate(item["input_images"]):
                # convert then exif_transpose, the order omnicontext/inference.py uses on every
                # input image. PIL does not apply tag 274 on open, and a 90/270 tag swaps the
                # stored width and height, so the transpose has to happen before size is recorded.
                orient[im.getexif().get(274, 1) or 1] += 1
                im = ImageOps.exif_transpose(im.convert("RGB"))
                rel = f"{cfg}/{key}_{j}.png"
                dst = os.path.join(IMGS, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if not os.path.exists(dst):
                    im.save(dst)
                paths.append(rel)
                sizes.append(list(im.size))
            # instruction_langugae is the field name in the dataset card; the official
            # scorer reads instruction_language. Accept either, record which was present.
            lang = item.get("instruction_language", item.get("instruction_langugae"))
            rows.append({"key": key, "config": cfg, "task_type": item["task_type"],
                         "instruction": item["instruction"], "instruction_language": lang,
                         "input_images": paths, "input_img_sizes": sizes})
        print(f"  {cfg}: {len(ds)} rows")
    with open(MANIFEST, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"wrote {MANIFEST}: {len(rows)} rows")

n_imgs = sum(len(r["input_images"]) for r in rows)
print(f"rows {len(rows)} | reference images {n_imgs} | keys unique {len(set(r['key'] for r in rows)) == len(rows)}")
if orient:
    print(f"EXIF orientation on the source images: {dict(sorted(orient.items()))} "
          f"(1 = none; non-trivial tags would have been applied by the transpose above)")

data-00000-of-00002.arrow: reconstructing file:   0%|          |  0.00B /  252MB            

data-00000-of-00002.arrow: downloading bytes:           |  0.00B            

data-00001-of-00002.arrow: reconstructing file:   0%|          |  0.00B /  420MB            

data-00001-of-00002.arrow: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

  default: 400 rows
wrote /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/omnicontext/manifest.jsonl: 400 rows
rows 400 | reference images 790 | keys unique True
EXIF orientation on the source images: {1: 790} (1 = none; non-trivial tags would have been applied by the transpose above)


## 3. Structure
Published shape is 8 subtasks x 50 = 400 across three primary categories.

In [3]:
by_task = collections.Counter(r["task_type"] for r in rows)
print(f"{'task_type':26s} {'rows':>5s}")
for t, c in sorted(by_task.items()):
    print(f"  {t:24s} {c:5d}")
print(f"  {'TOTAL':24s} {len(rows):5d}   [published 400 = 8 x 50]")
print(f"\nconfig == task_type on every row: {all(r['config'] == r['task_type'] for r in rows)}")

def primary(t):
    return "SCENE" if "scene" in t else ("MULTIPLE" if t.startswith("multi") else "SINGLE")
print("\nprimary category:", dict(collections.Counter(primary(r["task_type"]) for r in rows)))

task_type                   rows
  multi_character             50
  multi_character_object      50
  multi_object                50
  scene_character             50
  scene_character_object      50
  scene_object                50
  single_character            50
  single_object               50
  TOTAL                      400   [published 400 = 8 x 50]

config == task_type on every row: False

primary category: {'SINGLE': 100, 'MULTIPLE': 150, 'SCENE': 150}


## 4. Schema against the official scorer
`test_omnicontext_score.py` reads `task_type`, `instruction`, `input_images`, `key`. The dataset card documents the language field as `instruction_langugae`; the scorer reads `instruction_language`. Section 2 accepted either — this records which the pinned revision actually ships.

In [4]:
ds0 = load_dataset(REPO_ID, CONFIGS[0], split="train", revision=REVISION)
fields = list(ds0.features.keys())
print("fields shipped:", fields)
for want in ["task_type", "instruction", "input_images", "key"]:
    print(f"  scorer reads {want:16s} present: {want in fields}")
print(f"  language field: instruction_language={'instruction_language' in fields}  "
      f"instruction_langugae={'instruction_langugae' in fields}")
print(f"  language resolved on every row: {all(r['instruction_language'] is not None for r in rows)}")

fields shipped: ['task_type', 'instruction', 'input_images', 'instruction_language', 'key']
  scorer reads task_type        present: True
  scorer reads instruction      present: True
  scorer reads input_images     present: True
  scorer reads key              present: True
  language field: instruction_language=True  instruction_langugae=False
  language resolved on every row: True


## 5. with_scene split
`test_omnicontext_score.py`: `with_scene = item['task_type'].find('scene') != -1`. The flag switches both the PF and the SC rubric, not SC alone. The domain arm runs `with_scene=False` throughout because no domain row carries a scene image; the OmniContext scorer has to branch.

In [5]:
def with_scene(t):
    return t.find("scene") != -1

split = collections.Counter(with_scene(r["task_type"]) for r in rows)
print(f"with_scene=True  rows: {split[True]:3d}  subtasks: {sorted({r['task_type'] for r in rows if with_scene(r['task_type'])})}")
print(f"with_scene=False rows: {split[False]:3d}  subtasks: {sorted({r['task_type'] for r in rows if not with_scene(r['task_type'])})}")

with_scene=True  rows: 150  subtasks: ['scene_character', 'scene_character_object', 'scene_object']
with_scene=False rows: 250  subtasks: ['multi_character', 'multi_character_object', 'multi_object', 'single_character', 'single_object']


## 6. Instruction language
OmniGen2 reports degraded output on Chinese prompts (paper section 6, limitation 1). Any non-English rows are a confound the arms inherit and are stratified rather than dropped.

In [6]:
lang = collections.Counter(r["instruction_language"] for r in rows)
print("instruction_language:", dict(lang))
if len(lang) > 1:
    print("\nby task_type:")
    for t in sorted(by_task):
        c = collections.Counter(r["instruction_language"] for r in rows if r["task_type"] == t)
        print(f"  {t:24s} {dict(c)}")

instruction_language: {'en': 400}


## 7. Token budget
Every arm generates at `max_sequence_length=1024`. The domain test split measured 220 raw tokens at maximum against 245 templated, so the chat template costs about 25 rows; the check below applies that measured overhead. Truncation here would silently drop instruction text at generation time.

In [7]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.51.3"], check=True)
from transformers import AutoTokenizer
TOK = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

MAX_SEQ, TEMPLATE_OVERHEAD = 1024, 25
raw = [len(TOK(r["instruction"], add_special_tokens=False).input_ids) for r in rows]
over = [n + TEMPLATE_OVERHEAD for n in raw]
print(f"raw instruction tokens : mean {sum(raw)/len(raw):.0f}  max {max(raw)}")
print(f"+ template overhead    : max {max(over)}  against MAX_SEQ {MAX_SEQ}")
print(f"rows over the cap      : {sum(1 for n in over if n > MAX_SEQ)}/{len(rows)}")
print(f"headroom at the longest: {MAX_SEQ - max(over)} rows")
print("\nlongest row per task_type:")
for t in sorted(by_task):
    m = max(n for n, r in zip(over, rows) if r["task_type"] == t)
    print(f"  {t:24s} {m:5d}")

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

raw instruction tokens : mean 34  max 155
+ template overhead    : max 180  against MAX_SEQ 1024
rows over the cap      : 0/400
headroom at the longest: 844 rows

longest row per task_type:
  multi_character             82
  multi_character_object     141
  multi_object               113
  scene_character             72
  scene_character_object      88
  scene_object               112
  single_character           141
  single_object              180


## 8. Reference count and image sizes
Reference count is the axis reported alongside the domain headline, so both legs stratify identically. Sizes drive what `load_inputs(max_side=512)` does at generation and what the 1024^2 cap does at scoring.

In [8]:
import math
nref = collections.Counter(len(r["input_images"]) for r in rows)
print("references per row:", dict(sorted(nref.items())))
print("  domain test mix for comparison: {1: 243, 2: 194, 3: 51} of 488")
print("\nby task_type:")
for t in sorted(by_task):
    c = collections.Counter(len(r["input_images"]) for r in rows if r["task_type"] == t)
    print(f"  {t:24s} {dict(sorted(c.items()))}")

sizes = [tuple(s) for r in rows for s in r["input_img_sizes"]]
print(f"\ndistinct reference sizes: {len(set(sizes))}")
for s, c in collections.Counter(sizes).most_common(8):
    print(f"  {s[0]:5d} x {s[1]:5d}  {c:5d}")

def gen_side(w, h):
    # load_inputs(max_side=512), the resize every arm applies before the VAE
    s = 512 / max(w, h)
    W, H = (int(w * s), int(h * s)) if s < 1 else (w, h)
    return max(16, W // 16 * 16), max(16, H // 16 * 16)

def judge_side(w, h):
    # test_omnicontext_score.py: cap at 1024^2 then floor each side to /16
    if w * h > 1024 * 1024:
        r = math.sqrt(1024 * 1024 / (w * h)); w, h = int(w * r), int(h * r)
    return w // 16 * 16, h // 16 * 16

print("\nresize of the most common sizes:")
for s, c in collections.Counter(sizes).most_common(5):
    print(f"  {s[0]:5d}x{s[1]:<5d} -> generation {gen_side(*s)}   judge {judge_side(*s)}")
print(f"\nreferences above the judge cap (1024^2): "
      f"{sum(1 for w, h in sizes if w * h > 1024 * 1024)}/{len(sizes)}")

references per row: {1: 100, 2: 210, 3: 90}
  domain test mix for comparison: {1: 243, 2: 194, 3: 51} of 488

by task_type:
  multi_character          {2: 41, 3: 9}
  multi_character_object   {2: 49, 3: 1}
  multi_object             {2: 50}
  scene_character          {2: 23, 3: 27}
  scene_character_object   {3: 50}
  scene_object             {2: 47, 3: 3}
  single_character         {1: 50}
  single_object            {1: 50}

distinct reference sizes: 168
   1024 x   682    204
    682 x  1024    166
    768 x  1024     38
   1024 x   768     30
   1024 x   683     29
   1024 x  1024     23
   1024 x   680     18
    683 x  1024     16

resize of the most common sizes:
   1024x682   -> generation (512, 336)   judge (1024, 672)
    682x1024  -> generation (336, 512)   judge (672, 1024)
    768x1024  -> generation (384, 512)   judge (768, 1024)
   1024x768   -> generation (512, 384)   judge (1024, 768)
   1024x683   -> generation (512, 336)   judge (1024, 672)

references above the judge

## 9. Domain overlap
The same 17-keyword filter that built the domain subset, applied to the 400 instructions. OmniContext is the generalisation leg, so any in-domain rows are reported as a stratum rather than removed: a config that gains on them while losing elsewhere is the specialisation trade-off measured inside one benchmark.

In [9]:
import re
keywords = [
    "desktop computer","computer setup","laptop","tablet","smartphone",
    "mobile phone","cell phone","headphones","earphones","earbuds",
    "keyboard","computer mouse","game controller","gaming controller",
    "gaming console","vr headset","virtual reality headset",
]
pat = re.compile("|".join(re.escape(k) for k in keywords), re.IGNORECASE)
hits = [r for r in rows if pat.search(r["instruction"])]
print(f"rows matching >=1 domain keyword: {len(hits)}/{len(rows)} ({100*len(hits)/len(rows):.1f}%)")
if hits:
    print("\nby task_type:", dict(collections.Counter(r["task_type"] for r in hits)))
    print("which keywords fired:",
          dict(collections.Counter(k for r in hits for k in keywords
                                   if re.search(re.escape(k), r["instruction"], re.I)).most_common()))
    # musical keyboards are the known false positive; same check run on the domain subset
    mus = re.compile(r"\bpiano|musician|synthesi|keyboardist|orchestra|\bband\b|concert|guitar|drummer", re.I)
    fp = [r for r in hits if re.search(r"keyboard", r["instruction"], re.I) and mus.search(r["instruction"])
          and sum(1 for k in keywords if re.search(re.escape(k), r["instruction"], re.I)) == 1]
    print(f"of those, likely musical-keyboard false positives: {len(fp)}")
    print("\nfirst matches:")
    for r in hits[:5]:
        print(f"  [{r['task_type']}] {r['instruction'][:140]}")
else:
    print("no in-domain rows; OmniContext is a clean out-of-domain leg")

rows matching >=1 domain keyword: 14/400 (3.5%)

by task_type: {'single_character': 1, 'single_object': 3, 'multi_object': 2, 'multi_character_object': 1, 'scene_character': 2, 'scene_object': 2, 'scene_character_object': 3}
which keywords fired: {'laptop': 8, 'tablet': 3, 'keyboard': 2, 'earphones': 1, 'headphones': 1}
of those, likely musical-keyboard false positives: 0

first matches:
  [single_character] The young woman gracefully adjusts her delicate earrings while seated at a cozy desk, surrounded by books and a laptop, immersed in a moment
  [single_object] Let the small bird perch delicately on the edge of a rustic wooden table, its feathers subtly ruffled by a gentle breeze coming through the 
  [single_object] Position the classic Volkswagen van with a two-tone color scheme in a cozy workspace, parked beside a wooden desk adorned with an open lapto
  [single_object] Put the carved pumpkin on a sleek, minimalist workspace table made of matte black metal and pale wood. The glow

## 10. Summary

In [10]:
print(f"revision            {REVISION}")
print(f"rows                {len(rows)} across {len(by_task)} subtasks")
print(f"reference images    {n_imgs} extracted to {IMGS}")
print(f"with_scene split    {split[True]} True / {split[False]} False")
print(f"languages           {dict(lang)}")
print(f"token cap           max {max(over)} templated against {MAX_SEQ}, over-cap rows {sum(1 for n in over if n > MAX_SEQ)}")
print(f"reference mix       {dict(sorted(nref.items()))}")
print(f"in-domain rows      {len(hits)}")
print("\ngeneration config for every OmniContext arm, identical to the domain arms:")
print("  512x512, align_res=False, seed 0, 50 steps, text_g 5.0, image_g 2.0, max_sequence_length 1024")
print("  deviates from the shipped 1024x1024 align_res=True defaults; applied to every arm alike")

revision            0f6aabbe8d3448d51184093372e980205d9980d3
rows                400 across 8 subtasks
reference images    790 extracted to /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/omnicontext/images
with_scene split    150 True / 250 False
languages           {'en': 400}
token cap           max 180 templated against 1024, over-cap rows 0
reference mix       {1: 100, 2: 210, 3: 90}
in-domain rows      14

generation config for every OmniContext arm, identical to the domain arms:
  512x512, align_res=False, seed 0, 50 steps, text_g 5.0, image_g 2.0, max_sequence_length 1024
  deviates from the shipped 1024x1024 align_res=True defaults; applied to every arm alike
